In [ ]:
import operator
from functools import reduce

import polars as pl

from hexagonal.files.spec import get_polars_dataframe

from rapidfuzz import fuzz

In [ ]:
import polars.selectors as cs

In [ ]:
pl.Config.set_tbl_rows(20)
pl.Config.set_fmt_str_lengths(100)

In [ ]:
def normaliser(colonne):
    if isinstance(colonne, str):
        colonne = pl.col(colonne)
    return (
        colonne
        .str.normalize("NFKD")  # splitter les diacritiques
        .str.to_lowercase()
        .str.replace_all(r"œ", "oe")    # à traiter spécifiquement en français
        .str.replace_all(r"\p{Nonspacing Mark}", "")  # retirer les diacritiques
        .str.replace_all(r"\pP", " ")  # ponctuation
        .str.replace_all(r"\s\s+", " ")
        .str.strip_chars()
    )

In [ ]:
def scorer(a):
    return float(fuzz.ratio(*a.values()))

# Conditions

In [ ]:
maire_soutien = (pl.col("parrainage_2022") == "MÉLENCHON Jean-Luc").fill_null(False)
maire_corse = pl.col("code_commune").str.slice(0, 2).is_in(["2A", "2B"])
maire_nouvelle_caledonie = pl.col("code_commune").str.slice(0, 3) == "988"
maire_outremer = pl.col("code_commune").str.slice(0, 2).is_in(["97", "98"])

# Sources de données

## COG

In [ ]:
communes = get_polars_dataframe("data/03_main/cog/communes.csv")

In [ ]:
departements = get_polars_dataframe("data/02_clean/cog/departements.csv")

## Annuaire

In [ ]:
mairies = get_polars_dataframe("data/02_clean/annuaire/mairies.csv").with_columns(
    pl.col("emails").str.split("\n|,", literal=False)
)

In [ ]:
conseils_departementaux = get_polars_dataframe("data/02_clean/annuaire/conseils_departementaux.csv").with_columns(
    code_departement=pl.when(pl.col("code_departement") == "67").then(pl.lit("6AE")).otherwise("code_departement")
).select("code_departement", "nom", cs.starts_with("postale_")).with_columns(
    mandat=pl.when(
        pl.col("code_departement").is_in([
            "975", 
            "977",
            "978",
            "987",
            "988",
        ])
    ).then(pl.lit("membre assemblée outremer"))
    .otherwise(pl.lit("conseiller départemental"))
)

In [ ]:
conseils_regionaux = get_polars_dataframe("data/02_clean/annuaire/conseils_regionaux.csv").select(
    "code_departement",
    "nom",
    cs.starts_with("postale_")
).join(
    departements.select("code_departement", "code_region"),
    on=["code_departement"],
    how="left",
).select(
    pl.all().exclude("code_departement")
).join(
    pl.concat([
        departements.select("code_region", "code_departement"),
        pl.DataFrame({
            "code_region": ["44", "84", "94"],
            "code_departement": ["6AE", "69M", "20"]
        })
    ]),
    on=["code_region"],
    how="left"
).with_columns(
    mandat=pl.when(
        pl.col("code_departement").is_in(["972", "973"])
    ).then(
        pl.lit("membre assemblée outremer")
    ).when(
        pl.col("code_departement") == "20"
    ).then(
        pl.lit("membre assemblée corse")
    ).otherwise(pl.lit("conseiller régional"))
)

## RNE

In [ ]:
nom_complet = pl.format(
    "{} {} {}",
    pl.col("sexe").replace_strict(["M", "F"], ["M.", "Mme"]),
    "prenom",
    "nom"
).alias("nom_complet")

In [ ]:
maires = get_polars_dataframe("data/02_clean/rne/conseillers_municipaux.csv").filter(
    pl.col("fonction").is_in(["Maire", "Maire délégué"])
).select(
    "code_commune", 
    "nom",
    "prenom",
    "sexe",
    pl.col("fonction").str.to_lowercase(),
    nom_complet,
).sort(
    ["code_commune", "nom_complet", "fonction"]
).unique(["code_commune", "nom_complet"], keep="first")

In [ ]:
conseillers_departementaux = get_polars_dataframe("data/02_clean/rne/conseillers_departementaux.csv").unique(
    ["code_departement", "nom", "prenom", "sexe", "date_naissance"]
)

In [ ]:
conseillers_regionaux = get_polars_dataframe("data/02_clean/rne/conseillers_regionaux.csv").unique(
    ["code_departement", "nom", "prenom", "sexe", "date_naissance"]
)

In [ ]:
membres_csp = get_polars_dataframe("../../../data/02_clean/rne/conseillers_csp.csv").unique(
    ["code_csp", "nom", "prenom", "sexe", "date_naissance"]
)

## Parrainages

In [ ]:
parrainages = get_polars_dataframe("data/03_main/elections/2022-presidentielle-parrainages.csv").with_row_index().with_columns(
    index="CC" + pl.col("index").cast(pl.String()).str.zfill(5)
)

In [ ]:
parrainages_maires = parrainages.filter(
    pl.col("mandat").is_in(["maire", "maire délégué"])
).select(
    pl.col("code_circonscription").alias("code_commune"),
    "nom",
    "prenom",
    "sexe",
    "candidat",
    pl.col("mandat").alias("fonction"),
    nom_complet
)

In [ ]:
parrainages_elus_territoriaux = parrainages.filter(
    pl.col("mandat").is_in(["conseiller départemental", "conseiller régional", "membre assemblée outremer", "membre assemblée corse"])
).select(
    "index",
    pl.col("code_circonscription").replace(
        ["67", "68", "2A", "2B"],
        ["6AE", "6AE", "20", "20"]
    ).alias("code_departement"),
    "mandat",
    "nom",
    "prenom",
    "sexe", 
    "candidat",
    nom_complet
)

In [ ]:
emails_individuels = pl.read_csv("out/2026-06-emails-maires.csv", schema_overrides={"code_commune": pl.String()}).with_columns(
    nom_complet
)

# Préparer les données

## Identifier les maires qui ont été parrains en 2022

In [ ]:
exclusions = [
    ("17114", "Mme Marie-Noëlle SURAUD"),
    ("55100", "M. Loïc VINCENT"),
    ("61225", "M. Cyrille COTREL-LASSAUSSAYE"),
    ("30129", "M. Thierry ORTIZ"),
    ("61286", "Mme Marie-Claude CORVÉ"),
]

In [ ]:
maires_qualifies = maires.join(
    parrainages_maires.select("code_commune", "fonction", "nom_complet", "candidat"), 
    on=["code_commune"],
    how="left",
    suffix="_parrainage"
).with_columns(
    score=pl.when(
        pl.col("nom_complet_parrainage").is_not_null()
    ).then(
        pl.struct(normaliser("nom_complet"), normaliser("nom_complet_parrainage")).map_elements(
            scorer,
            return_dtype=pl.Float64(),
            skip_nulls=True,
        )
    )
).sort(
    ["code_commune", "nom_complet", "score"],
    descending=[False, False, True]
).unique(
    ["code_commune", "nom_complet"]
).select(
    *maires.columns,
    parrainage_2022=pl.when(
        (pl.col("score") >= 80.) & ~pl.struct("code_commune", "nom_complet").is_in(
            pl.Series(exclusions, dtype=pl.Struct({"code_commune": pl.String(), "nom_complet": pl.String()})).implode()
        )
    ).then("candidat")
)

## Identifier les conseillers départementaux, conseillers régionaux et conseillers territoriaux parrains en 2022

In [ ]:
import splink.exploratory as exp
import splink.comparison_level_library as cll
import splink.comparison_library as cl
from splink import DuckDBAPI, block_on, SettingsCreator, Linker

In [ ]:
elus_territoriaux = pl.concat(
    [
        conseillers_departementaux.select(
            "code_departement",
            "nom",
            "prenom",
            "sexe",
            pl.lit("conseiller départemental").alias("mandat")
        ),
        conseillers_regionaux.select(
            pl.col("code_departement").replace(["69M"], ["69"]),
            "nom",
            "prenom",
            "sexe",
            pl.lit("conseiller régional").alias("mandat")
        ),
        membres_csp.filter(
            ~pl.col("code_csp").is_in(["69M", "988C"])
        ).select(
            pl.col("code_csp").str.slice(0, pl.col("code_csp").str.len_chars() - 1).alias("code_departement"),
            "nom",
            "prenom",
            "sexe",
            pl.when(pl.col("code_csp") == "20R").then(pl.lit("membre assemblée corse")).otherwise(pl.lit("membre assemblée outremer")).alias("mandat")
        )
    ]
).with_row_index().with_columns(index="RNE" + pl.col("index").cast(pl.String()).str.zfill(4))

In [ ]:
noms = [
    normaliser("prenom"),
    normaliser("nom"),
    pl.format("{} {}", normaliser("prenom"), normaliser("nom")).alias("nom_complet")
]

tables = [
    elus_territoriaux.with_columns(*noms).to_pandas(),
    parrainages_elus_territoriaux.select(
        "index",
        "code_departement",
        *noms,
        "sexe",
        "mandat"
    ).to_pandas()
]

In [ ]:
blocks = [
    block_on("prenom", "nom"),
    block_on("code_departement"),
    block_on("mandat"),
]

full_name_comparison = {
    "output_column_name": "Nom complet",
    "comparison_levels": [
        {
            "sql_condition": "(prenom_l IS NULL OR prenom_r IS NULL) and (nom_l IS NULL OR nom_r IS NULL)",
            "label_for_charts": "Null",
            "is_null_level": True,
        },
        cll.ExactMatchLevel("nom_complet", term_frequency_adjustments=True),
        cll.JaroWinklerLevel("nom_complet", 0.9),
        cll.ColumnsReversedLevel("nom", "prenom"),
        {
            "sql_condition": "jaro_winkler_similarity(prenom_l, nom_r) + jaro_winkler_similarity(nom_l, prenom_r) >= 1.8",
            "label_for_charts": "switched + jaro_winkler_similarity >= 1.8",
        },
        {
            "sql_condition": """
                (
                    contains(prenom_l, prenom_r) or contains(prenom_r, prenom_l)
                ) and (
                    contains(nom_l, nom_r) or contains(nom_r, nom_l)
                )""",
            "label_for_charts": "contained"
        },
        cll.ExactMatchLevel("prenom", term_frequency_adjustments=True),
        cll.ExactMatchLevel("nom", term_frequency_adjustments=True),
        {
            "sql_condition": "prenom_l = nom_r or nom_l = prenom_r",
            "label_for_charts": "single name cross-match"
        },
        cll.JaroWinklerLevel("prenom", 0.9),
        cll.JaroWinklerLevel("nom", 0.9),
        cll.ElseLevel(),
    ]
}

comparisons = [
    full_name_comparison,
    cl.ExactMatch("sexe").configure(term_frequency_adjustments=True),
    cl.ExactMatch("code_departement").configure(term_frequency_adjustments=True),
    cl.ExactMatch("mandat")
]

settings = SettingsCreator(
    link_type="link_only",
    unique_id_column_name="index",
    blocking_rules_to_generate_predictions=blocks,
    comparisons=comparisons,
    retain_intermediate_calculation_columns=True,
)

linker = Linker(
    tables,
    settings,
    DuckDBAPI()
)

In [ ]:
deterministic_rules = [
    block_on("nom", "prenom", "sexe"),
]

linker.training.estimate_probability_two_random_records_match(
    deterministic_rules,
    recall=0.95
)

In [ ]:
linker.training.estimate_u_using_random_sampling(1e7)

In [ ]:
session_departement = linker.training.estimate_parameters_using_expectation_maximisation(
    block_on("code_departement"), estimate_without_term_frequencies=True
)

In [ ]:
session_mandat = linker.training.estimate_parameters_using_expectation_maximisation(
    block_on("mandat"), estimate_without_term_frequencies=True
)

In [ ]:
predictions = linker.inference.predict(threshold_match_probability=0.2)

In [ ]:
linker.visualisations.comparison_viewer_dashboard(
    predictions,
    "test.html"
)

In [ ]:
correspondances_elus_territoriaux = predictions.as_duckdbpyrelation().pl().filter(
    pl.col("match_probability") > .95
).select(
    index_rne="index_l",
    index_parrainage="index_r",
)

In [ ]:
elus_parrainage = elus_territoriaux.join(
    correspondances_elus_territoriaux,
    left_on=["index"],
    right_on=["index_rne"],
    how="left",
    maintain_order="left"
).join(
    parrainages_elus_territoriaux.select(
        "index",
        "candidat",
    ),
    left_on=["index_parrainage"],
    right_on=["index"],
    how="left",
    maintain_order="left",
).select(
    "index",
    "code_departement",
    "mandat",
    nom_complet,
    "sexe",
    pl.col("candidat").alias("parrainage_2022")
)

## Identifier la mairie principale

In [ ]:
dedup = [
    "d268cedd-dc8c-4204-9226-ef3ff576d91d",
    "6cf098c7-53d0-4e19-9d95-15b0bc89d3e9",
    "873a2bf8-aaec-4c1e-a0bb-177c74b0a2e4",
    "b013c5c8-7f0c-4afe-9052-6a0233092c38",
    "15944e99-e55a-475a-8132-5e160995799e",
    "deae784b-d2c9-4527-aee0-55ddf2939288",
    "ca05a12a-7a3a-475a-9d75-3945467733aa",
    "2ee8945b-8707-479d-b9a9-f0235df112f1",
    "3e52c219-b9f4-4150-b517-f1bc0a5f4e41",
    "8bb65ac2-f71f-4ddf-942d-1a363cb32983",
    "34f09172-217b-4c03-932b-cbf47f2117c2",
    "568a88ff-21dc-4d10-b166-fdcc6086bfd9",
    "114a1668-d6d8-4809-b8f6-2f5e29acfcab",
    "af20c30b-5cc7-4b1e-b7f6-65d766a4b503"
]

In [ ]:
sans_mairie_deleguee = ~pl.col("nom").str.contains(r"(?i)(déléguéé?e|annexe)")
unique = pl.col("nom").count().over("code_commune") == 1
nom_correspond = normaliser("nom").str.contains("^mairie " + normaliser("physique_commune")+"$")
in_dedup = pl.col("id").is_in(dedup)

In [ ]:
mairies_principales = mairies.filter(
    sans_mairie_deleguee
).filter(
    unique | in_dedup | (nom_correspond & ~in_dedup.any().over("code_commune"))
).sort("code_commune")

## Traiter les conseils départementaux et régionaux

In [ ]:
additionnels = [
    {
        "code_departement": "986",
        "nom": "Assemblée territoriale de Wallis-et-Futuna",
        "postale_service_distribution": "BP 31 Mata'Utu - Havelu Hahake",
        "postale_code_postal": "98600",
        "postale_commune": "Uvea",
        "mandat": "membre assemblée outremer"
    },
    {
        "code_departement": "987",
        "nom": "Assemblée de la Polynésie française",
        "postale_service_distribution": "BP 28",
        "postale_commune": "Papeete",
        "postale_code_postal": "98713",
        "mandat": "membre assemblée outremer",
    }
]

In [ ]:
conseils = pl.concat(
    [
        conseils_departementaux,
        conseils_regionaux.select("code_departement", "nom", cs.starts_with("postale_"), "mandat"),
        pl.from_records(additionnels, orient="row"),
    ],
    how="diagonal"
).unique("code_departement", keep="last")

## Vérifier qu'on a bien tout

In [ ]:
maires_qualifies.filter(~pl.col("code_commune").is_in(mairies_principales["code_commune"].implode()))

# Préparer les courriers papier

## Maires

In [ ]:
champ_lieu = (
    pl.when(
        pl.col("code_commune").str.slice(0, 2).is_in(["2A", "2B"])
    ).then(
        pl.lit("corse")
    ).when(
        pl.col("code_commune").str.slice(0, 3) == "988"
    ).then(
        pl.lit("nouvelle-caledonie")
    ).when(
        pl.col("code_commune").str.slice(0, 2).is_in(["97", "98"])
    ).then(
        pl.lit("outremer")
    )
)

champ_parrains = (
    pl.when(
        pl.col("parrainage_2022") == "MÉLENCHON Jean-Luc"
    ).then(
        pl.lit("parrains")
    )
)

In [ ]:
adresses = maires_qualifies.select(
    "code_commune", "nom_complet", "sexe", "fonction", "parrainage_2022"
).join(
    mairies_principales.select(
        "code_commune",
        pl.col("nom").alias("postale_nom"),
        *(c for c in mairies_principales.columns if c.startswith("postale_"))
    ),
    on=["code_commune"],
    how="inner"
).with_columns(
    courrier=pl.concat_str(
        pl.lit("maires"),
        champ_lieu,
        champ_parrains,
        separator="-",
        ignore_nulls=True,
    )
)

In [ ]:
adresses["courrier"].value_counts(sort=True)

In [ ]:
adresses.filter(pl.col("courrier") != "maires").select(
    pl.col("code_commune").alias("code"),
    "nom_complet",
    "sexe",
    "courrier",
    pl.concat_str(
        pl.col("nom_complet").str.to_uppercase(),
        pl.lit("Mairie").str.to_uppercase(),
        pl.col("postale_numero_voie").str.to_uppercase(),
        pl.col("postale_complement1").str.to_uppercase(),
        pl.col("postale_complement2").str.to_uppercase(),
        pl.col("postale_service_distribution").str.to_uppercase(),
        pl.format("{} {}", "postale_code_postal", "postale_commune").str.to_uppercase(),
        separator="\n",
        ignore_nulls=True,
    ).alias("adresse_complete"),
).write_csv("out/adresses_maires_speciaux.csv")

## Autres élus

In [ ]:
lieux_speciaux = {
    "20": "corse",
    "971": "outremer",
    "972": "martinique",
    "973": "guyane",
    "974": "outremer",
    "975": "saint-pierre-et-miquelon",
    "976": "outremer",
    "977": "saint-barthelemy",
    "978": "saint-martin",
    "986": "wallis-et-futuna",
    "987": "polynesie",
    "988": "nouvelle-caledonie",
}

In [ ]:
courriers_elus_territoriaux = elus_parrainage.join(
    conseils,
    on=["code_departement", "mandat"],
    how="left"
).with_columns(
    courrier=pl.concat_str(
        [
            pl.col("mandat").replace_strict(
                ["conseiller départemental", "conseiller régional", "membre assemblée outremer", "membre assemblée corse"],
                ["cd", "cr", "membres", "membres"]
            ),
            pl.col("code_departement").replace_strict(
                list(lieux_speciaux.keys()),
                list(lieux_speciaux.values()),
                default=pl.lit(None),
            ),
            pl.when(pl.col("parrainage_2022") == "MÉLENCHON Jean-Luc").then(pl.lit("parrains"))
        ],
        separator="-",
        ignore_nulls=True
    )
).select(
    pl.col("code_departement").alias("code"),
    "nom_complet",
    "sexe",
    "courrier",
    pl.concat_str(
        pl.col("nom_complet").str.to_uppercase(),
        pl.col("nom").str.to_uppercase(),
        pl.col("postale_numero_voie").str.to_uppercase(),
        pl.col("postale_complement1").str.to_uppercase(),
        pl.col("postale_complement2").str.to_uppercase(),
        pl.col("postale_service_distribution").str.to_uppercase(),
        pl.format("{} {}", "postale_code_postal", "postale_commune").str.to_uppercase(),
        separator="\n",
        ignore_nulls=True,
    ).alias("adresse_complete"),
)

In [ ]:
courriers_elus_territoriaux.write_csv("out/adresses_territoriaux.csv")

# Préparer les envois emails

In [ ]:
emails_mairies = mairies_principales.select(
    "code_commune", "emails"
).filter(pl.col("emails").is_not_null())

In [ ]:
envois_mairies = maires_qualifies.join(
    emails_mairies,
    on=["code_commune"],
).sort(["code_commune", "fonction"]).explode("emails").unique(["emails"], keep="first")

In [ ]:
envois_individuels = maires_qualifies.join(
    emails_individuels,
    on=["code_commune"],
    suffix="_email"
).with_columns(
    score=pl.struct(normaliser("nom_complet"), normaliser("nom_complet_email")).map_elements(
        lambda s: fuzz.token_ratio(*s.values()),
        return_dtype=pl.Float64(),
        skip_nulls=True,
    )
).filter(pl.col("score") == 100.)

In [ ]:
envois_emails = pl.concat([
    envois_mairies.select(
        "code_commune", "nom", "prenom", "sexe", "fonction", "parrainage_2022", pl.col("emails").alias("email")
    ),
    envois_individuels.select(
        "code_commune", "nom", "prenom", "sexe", "fonction", "parrainage_2022", "email"
    )
])

In [ ]:
def pour_envoi(emails, chemin):
    emails.select(
        "email",
        pl.format("{prenom} {nom}").alias("name"),
        pl.struct("code_commune", "fonction", "parrainage_2022", "sexe").struct.json_encode().alias("attributes")
    ).write_csv(chemin) 

In [ ]:
echantillons = {
    "corse": pl.col("code_commune").str.slice(0, 2).is_in(["2A", "2B"]),
    "nouvelle-calédonie": pl.col("code_commune").str.slice(0, 3) == "988",
    "outremer": pl.col("code_commune").str.slice(0, 2) == "97",
}

echantillons["hexagone"] = ~reduce(operator.or_, echantillons.values())

In [ ]:
for nom, cond in echantillons.items():
    pour_envoi(
        envois_emails.filter(cond),
        f"emails_{nom}.csv"
    )

# Divers